In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

import socceraction.spadl as spadl
import socceraction.vaep.features as fs
import socceraction.vaep.labels as lab

spadl_h5 = "data/spadl-statsbomb.h5"

with pd.HDFStore(spadl_h5) as store:
    games = store["games"]
    players = store["players"]
    teams = store["teams"]
    player_games = store["player_games"]

print(f"games to process: {len(games)}")

games to process: 64


In [2]:
def load_and_fix_game(game_id, store):
    actions = store[f"actions/game_{game_id}"]
    actions = (
        actions
        .merge(spadl.actiontypes_df(), how="left")
        .merge(spadl.results_df(), how="left")
        .merge(spadl.bodyparts_df(), how="left")
    )
    # Fix flipped penalty coordinates
    pen_mask = (actions['type_name'] == 'shot_penalty') & (actions['start_x'] < 50)
    if pen_mask.any():
        actions.loc[pen_mask, 'start_x'] = 105 - actions.loc[pen_mask, 'start_x']
        actions.loc[pen_mask, 'start_y'] = 68 - actions.loc[pen_mask, 'start_y']
        actions.loc[pen_mask, 'end_x'] = 105 - actions.loc[pen_mask, 'end_x']
        actions.loc[pen_mask, 'end_y'] = 68 - actions.loc[pen_mask, 'end_y']
    return actions

In [3]:
feature_generators = [
    fs.actiontype_onehot,
    fs.bodypart_onehot,
    fs.result_onehot,
    fs.startlocation,
    fs.endlocation,
    fs.movement,
    fs.space_delta,
    fs.startpolar,
    fs.endpolar,
    fs.team,
    fs.time_delta
]

all_features = []
all_labels = []

for game in games.itertuples():
    game_id = game.game_id

    with pd.HDFStore(spadl_h5) as store:
        actions = load_and_fix_game(game_id, store)
    # # === COORDINATE FIX FOR FLIPPED PENALTIES ===
    # pen_mask = (actions['type_name'] == 'shot_penalty') & (actions['start_x'] < 50)
    # if pen_mask.any():
    #     n_fixed = pen_mask.sum()
    #     actions.loc[pen_mask, 'start_x'] = 105 - actions.loc[pen_mask, 'start_x']
    #     actions.loc[pen_mask, 'start_y'] = 68 - actions.loc[pen_mask, 'start_y']
    #     actions.loc[pen_mask, 'end_x'] = 105 - actions.loc[pen_mask, 'end_x']
    #     actions.loc[pen_mask, 'end_y'] = 68 - actions.loc[pen_mask, 'end_y']
    #     print(f"  Game {game_id}: flipped {n_fixed} penalty coordinate(s)")
    # # ============================================

    
    gamestates = fs.gamestates(actions, nb_prev_actions=3)

    game_features = pd.concat(
        [fn(gamestates) for fn in feature_generators],
        axis=1
    )
    game_features["game_id"] = game_id
    all_features.append(game_features)

   
    labelfns=[lab.scores, lab.concedes]
    game_labels = pd.concat([fn(actions) for fn in labelfns], axis=1)
    game_labels["game_id"] = game_id
    all_labels.append(game_labels)

features = pd.concat(all_features).reset_index(drop=True)
labels = pd.concat(all_labels).reset_index(drop=True)

print(f"features matrix shape: {features.shape}")
print(f"labels matrix shape: {labels.shape}")
print(f"\nLabel columns: {labels.columns.tolist()}")
print(f"\nScoring rate (fraction of actions where teams score next): {labels['scores'].mean():.4f}")
print(f"Conceding rate (fraction of actions where teams concede next): {labels['concedes'].mean():.4f}")

features matrix shape: (138944, 143)
labels matrix shape: (138944, 3)

Label columns: ['scores', 'concedes', 'game_id']

Scoring rate (fraction of actions where teams score next): 0.0107
Conceding rate (fraction of actions where teams concede next): 0.0024


In [4]:
with pd.HDFStore(spadl_h5) as store:
    test_actions = load_and_fix_game(3869685, store)

# Show all penalties from this match  
pens = test_actions[test_actions['type_name'] == 'shot_penalty']
print(f"Penalties in WC final: {len(pens)}")
print(pens[['type_name', 'result_name', 'start_x', 'start_y', 'end_x', 'end_y']].to_string(index=False))


Penalties in WC final: 11
   type_name result_name  start_x  start_y    end_x   end_y
shot_penalty     success  94.0625   34.425 104.5625 32.5125
shot_penalty     success  94.0625   34.425 104.5625 36.3375
shot_penalty     success  94.0625   34.425 104.5625 36.8475
shot_penalty     success  94.1500   34.340 104.5625 36.0825
shot_penalty     success  94.1500   34.340 104.5625 35.4875
shot_penalty        fail  94.1500   34.340 103.5125 35.5725
shot_penalty     success  94.1500   34.340 104.5625 33.7025
shot_penalty        fail  94.1500   34.340 104.5625 37.7825
shot_penalty     success  94.1500   34.340 104.5625 36.0825
shot_penalty     success  94.1500   34.340 104.5625 34.6375
shot_penalty     success  94.1500   34.340 104.5625 35.8275


## Training a VAEP Model

In [5]:
import optuna
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split

feature_cols = [c for c in features.columns if c != "game_id"]
X = features[feature_cols]

all_game_ids = games["game_id"].values

train_ids, test_ids = train_test_split(all_game_ids, test_size=0.2, random_state=42)

train_mask = features["game_id"].isin(train_ids)
test_mask = features["game_id"].isin(test_ids)

print(f"Train actions: {train_mask.sum()}, Test actions: {test_mask.sum()}")

models = {}
def objective(trial):
    n_estimators = trial.suggest_int("n_estimators", 50, 300)
    max_depth = trial.suggest_int("max_depth", 2, 20)
    learning_rate = trial.suggest_float("learning_rate", 0.01, 0.3)
    subsample = trial.suggest_float("subsample", 0.5, 1.0)
    colsample_bytree = trial.suggest_float("colsample_bytree", 0.5, 1.0)
    for target in ["scores", "concedes"]:
        print(f"\nTraining model for target: {target}")

        model = XGBClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            learning_rate=learning_rate,
            subsample=subsample,
            colsample_bytree=colsample_bytree,
            random_state=42,
            eval_metric="logloss",
        )
        model.fit(X[train_mask], labels[train_mask][target])

        from sklearn.metrics import brier_score_loss, roc_auc_score
        test_pred = model.predict_proba(X[test_mask])[:, 1]
        brier = brier_score_loss(labels[test_mask][target], test_pred)
        auc = roc_auc_score(labels[test_mask][target], test_pred)
        print(f"Brier score: {brier:.4f}, AUC: {auc:.4f}")

        return auc

study = optuna.create_study(
    direction="maximize", 
    sampler=optuna.samplers.RandomSampler(seed=42),
    study_name="rf_random_sampler"
)
study.optimize(objective, n_trials=30)


print("Best Trial Number:", study.best_trial.number)
print("Best Validation Accuracy:", study.best_value)
print("Best Hyperparameters:", study.best_params)


[I 2026-06-09 23:11:41,187] A new study created in memory with name: rf_random_sampler


Train actions: 111533, Test actions: 27411

Training model for target: scores


[I 2026-06-09 23:11:48,556] Trial 0 finished with value: 0.7098167615455361 and parameters: {'n_estimators': 144, 'max_depth': 20, 'learning_rate': 0.22227824312530747, 'subsample': 0.7993292420985183, 'colsample_bytree': 0.5780093202212182}. Best is trial 0 with value: 0.7098167615455361.


Brier score: 0.0093, AUC: 0.7098

Training model for target: scores


[I 2026-06-09 23:11:49,903] Trial 1 finished with value: 0.7220860804226152 and parameters: {'n_estimators': 89, 'max_depth': 3, 'learning_rate': 0.2611910822747312, 'subsample': 0.8005575058716043, 'colsample_bytree': 0.8540362888980227}. Best is trial 1 with value: 0.7220860804226152.


...

Brier score: 0.0094, AUC: 0.6910

Training model for target: scores


[I 2026-06-09 23:14:04,023] Trial 28 finished with value: 0.7167902614310605 and parameters: {'n_estimators': 291, 'max_depth': 6, 'learning_rate': 0.15420206670879177, 'subsample': 0.6504391549083848, 'colsample_bytree': 0.6424202471887338}. Best is trial 8 with value: 0.7425366483376472.


Brier score: 0.0093, AUC: 0.7168

Training model for target: scores


[I 2026-06-09 23:14:06,724] Trial 29 finished with value: 0.6935045162859224 and parameters: {'n_estimators': 59, 'max_depth': 13, 'learning_rate': 0.15577691673636984, 'subsample': 0.5257393756249946, 'colsample_bytree': 0.6393232321183058}. Best is trial 8 with value: 0.7425366483376472.


Brier score: 0.0093, AUC: 0.6935
Best Trial Number: 8
Best Validation Accuracy: 0.7425366483376472
Best Hyperparameters: {'n_estimators': 80, 'max_depth': 11, 'learning_rate': 0.019972671123413333, 'subsample': 0.954660201039391, 'colsample_bytree': 0.6293899908000085}


In [6]:
best_params = study.best_params

for target in ["scores", "concedes"]:
    model = XGBClassifier(
        n_estimators=best_params["n_estimators"],
        max_depth=best_params["max_depth"],
        learning_rate=best_params["learning_rate"],
        subsample=best_params["subsample"],
        colsample_bytree=best_params["colsample_bytree"],
        random_state=42,
        eval_metric="logloss",
    )
    model.fit(X[train_mask], labels[train_mask][target])
    models[target] = model
    from sklearn.metrics import brier_score_loss, roc_auc_score
    test_pred = model.predict_proba(X[test_mask])[:, 1]
    brier = brier_score_loss(labels[test_mask][target], test_pred)
    auc = roc_auc_score(labels[test_mask][target], test_pred)
    print(f"Trained final model for: {target}")
    print(f"Brier score: {brier:.4f}, AUC: {auc:.4f}")

Trained final model for: scores
Brier score: 0.0091, AUC: 0.7425
Trained final model for: concedes
Brier score: 0.0016, AUC: 0.6776


In [7]:
feature_all = X

P_scores = models["scores"].predict_proba(feature_all)[:, 1]
P_concedes = models["concedes"].predict_proba(feature_all)[:, 1]



all_actions_list = []
for game in games.itertuples():
    with pd.HDFStore(spadl_h5) as store:
       a = load_and_fix_game(game.game_id, store)
    # # === MUST APPLY THE SAME FIX HERE TOO ===
    # pen_mask = (a['type_name'] == 'shot_penalty') & (a['start_x'] < 50)
    # if pen_mask.any():
    #     a.loc[pen_mask, 'start_x'] = 105 - a.loc[pen_mask, 'start_x']
    #     a.loc[pen_mask, 'start_y'] = 68 - a.loc[pen_mask, 'start_y']
    #     a.loc[pen_mask, 'end_x'] = 105 - a.loc[pen_mask, 'end_x']
    #     a.loc[pen_mask, 'end_y'] = 68 - a.loc[pen_mask, 'end_y']
    # # ========================================

    a["game_id"] = game.game_id
    all_actions_list.append(a)

all_actions = pd.concat(all_actions_list).reset_index(drop=True)

assert len(all_actions) == len(feature_all), "Mismatch between actions and features length"

all_actions["P_scores"] = P_scores
all_actions["P_concedes"] = P_concedes

In [8]:
from socceraction.vaep import formula as vaepformula

vaep_values = vaepformula.value(
    all_actions,
    all_actions["P_scores"],
    all_actions["P_concedes"]
)

all_actions["vaep_value"] = vaep_values["offensive_value"] + vaep_values["defensive_value"]

print(f"Mean VAEP value: {all_actions['vaep_value'].mean():.4f}")
print(f"Max VAEP single action: {all_actions['vaep_value'].max():.4f}")
print(f"Min VAEP single action: {all_actions['vaep_value'].min():.4f}")

print(all_actions[["type_name", "P_scores", "P_concedes", "vaep_value"]].head(10))

Mean VAEP value: 0.0014
Max VAEP single action: 0.7667
Min VAEP single action: -1.1736
  type_name  P_scores  P_concedes  vaep_value
0      pass  0.005509    0.001432    0.000000
1   dribble  0.004791    0.001080   -0.000365
2      pass  0.012152    0.000747    0.007693
3   dribble  0.008739    0.000813   -0.003479
4      pass  0.011357    0.000759    0.002672
5   dribble  0.004357    0.000813   -0.007054
6      pass  0.010251    0.002107    0.004601
7   dribble  0.008438    0.000769   -0.000476
8      pass  0.005616    0.001062   -0.003115
9   dribble  0.005682    0.001075    0.000053


In [9]:
# Total minutes per player
minutes_per_player = (
    player_games.groupby('player_id')['minutes_played']
    .sum()
    .reset_index(name='total_minutes')
)

# Aggregate VAEP per player
vaep_by_player = (
    all_actions
    .groupby('player_id')
    .agg(
        total_vaep=('vaep_value', 'sum'),
        actions=('vaep_value', 'count'),
    )
    .reset_index()
)

# Merge minutes and names
vaep_by_player = (
    vaep_by_player
    .merge(minutes_per_player, on='player_id', how='left')
    .merge(players[['player_id', 'player_name', 'nickname']], on='player_id', how='left')
)
vaep_by_player['display_name'] = vaep_by_player.apply(
    lambda r: r['nickname'] if pd.notna(r['nickname']) and r['nickname'] else r['player_name'],
    axis=1
)

# Per-90 metric
vaep_by_player['vaep_per_90'] = (
    vaep_by_player['total_vaep'] / vaep_by_player['total_minutes'] * 90
)

# Qualify players with 270+ minutes
qualified = vaep_by_player[vaep_by_player['total_minutes'] >= 270].copy()

print("=== TOP 15 by TOTAL VAEP ===")
print(
    qualified
    .sort_values('total_vaep', ascending=False)
    .head(15)[['display_name', 'total_minutes', 'actions', 'total_vaep', 'vaep_per_90']]
    .to_string(index=False)
)

print("\n\n=== TOP 15 by VAEP PER 90 ===")
print(
    qualified
    .sort_values('vaep_per_90', ascending=False)
    .head(15)[['display_name', 'total_minutes', 'actions', 'total_vaep', 'vaep_per_90']]
    .to_string(index=False)
)

=== TOP 15 by TOTAL VAEP ===
   display_name  total_minutes  actions  total_vaep  vaep_per_90
  Kylian Mbappé            686      658    3.964861     0.520171
 Julián Álvarez            512      293    2.531266     0.444949
 Joško Gvardiol            749     1033    2.407241     0.289255
 Theo Hernández            571      656    2.258241     0.355940
    Bukayo Saka            311      265    2.077206     0.601121
     Cody Gakpo            484      353    1.758527     0.326999
     Nathan Aké            532      681    1.751598     0.296323
    Daley Blind            467      563    1.701607     0.327933
Frenkie de Jong            525      695    1.687724     0.289324
Jude Bellingham            467      565    1.638276     0.315728
 Olivier Giroud            447      151    1.624563     0.327093
  Nahuel Molina            632      652    1.527989     0.217593
Vinícius Júnior            311      267    1.520835     0.440113
    Richarlison            333      169    1.396433     0.377

In [10]:
print(qualified[qualified['display_name'].str.contains('Messi', na=False)][
    ['display_name', 'total_minutes', 'actions', 'total_vaep', 'vaep_per_90']
])

# And his ranks
total_rank = qualified.sort_values('total_vaep', ascending=False).reset_index(drop=True)
per90_rank = qualified.sort_values('vaep_per_90', ascending=False).reset_index(drop=True)
print(f"\nMessi rank by total VAEP: {total_rank[total_rank['display_name'].str.contains('Messi', na=False)].index.values + 1}")
print(f"Messi rank by VAEP per 90: {per90_rank[per90_rank['display_name'].str.contains('Messi', na=False)].index.values + 1}")

     display_name  total_minutes  actions  total_vaep  vaep_per_90
160  Lionel Messi            786      873    0.094979     0.010875

Messi rank by total VAEP: [211]
Messi rank by VAEP per 90: [217]


In [11]:
# Step 1: Identify shootout penalties (out-of-game-time penalties)
# In StatsBomb data, shootout penalties happen in period_id == 5
# Verify this column exists
print(f"Period IDs in data: {sorted(all_actions['period_id'].unique())}")
print(f"Penalties by period:")
print(all_actions[all_actions['type_name'] == 'shot_penalty']['period_id'].value_counts())

Period IDs in data: [1, 2, 3, 4, 5]
Penalties by period:
period_id
5    41
1    12
2    10
4     1
Name: count, dtype: int64


In [12]:
# Step 2: Set shootout penalty VAEP to 0 (they don't belong in the metric)
shootout_mask = (all_actions['type_name'] == 'shot_penalty') & (all_actions['period_id'] == 5)
print(f"Shootout penalties to zero out: {shootout_mask.sum()}")
all_actions.loc[shootout_mask, 'vaep_value'] = 0.0

# Step 3: Set regulation penalty VAEP to canonical values
regulation_pen_success = (
    (all_actions['type_name'] == 'shot_penalty') & 
    (all_actions['result_name'] == 'success') &
    (all_actions['period_id'] != 5)
)
regulation_pen_fail = (
    (all_actions['type_name'] == 'shot_penalty') & 
    (all_actions['result_name'] != 'success') &
    (all_actions['period_id'] != 5)
)

print(f"Regulation successful penalties: {regulation_pen_success.sum()}")
print(f"Regulation failed penalties: {regulation_pen_fail.sum()}")

all_actions.loc[regulation_pen_success, 'vaep_value'] = 0.76
all_actions.loc[regulation_pen_fail, 'vaep_value'] = -0.05

# Step 4: Re-aggregate
# Total minutes per player
minutes_per_player = (
    player_games.groupby("player_id", as_index=False)["minutes_played"]
    .sum()
    .rename(columns={"minutes_played": "total_minutes"})
)

# Aggregate VAEP per player
vaep_by_player = (
    all_actions
    .groupby("player_id", as_index=False)
    .agg(
        total_vaep=("vaep_value", "sum"),
        actions=("vaep_value", "count"),
    )
    .merge(minutes_per_player, on="player_id", how="left")
    .merge(players[["player_id", "player_name", "nickname"]], on="player_id", how="left")
)
vaep_by_player['display_name'] = vaep_by_player.apply(
    lambda r: r['nickname'] if pd.notna(r['nickname']) and r['nickname'] else r['player_name'],
    axis=1
)
vaep_by_player['vaep_per_90'] = (
    vaep_by_player['total_vaep'] / vaep_by_player['total_minutes'] * 90
)

qualified = vaep_by_player[vaep_by_player['total_minutes'] >= 270].copy()

# Where's Messi now?
total_rank = qualified.sort_values('total_vaep', ascending=False).reset_index(drop=True)
messi_row = total_rank[total_rank['display_name'].str.contains('Messi', na=False)]
print(f"\nMessi rank by total VAEP: {messi_row.index.values + 1}")
print(messi_row[['display_name', 'total_minutes', 'total_vaep', 'vaep_per_90']])

print("\n=== TOP 15 by TOTAL VAEP (final) ===")
print(
    total_rank
    .head(15)[['display_name', 'total_minutes', 'actions', 'total_vaep', 'vaep_per_90']]
    .to_string(index=False)
)

Shootout penalties to zero out: 41
Regulation successful penalties: 17
Regulation failed penalties: 6

Messi rank by total VAEP: [2]
   display_name  total_minutes  total_vaep  vaep_per_90
1  Lionel Messi            786    5.354761     0.613141

=== TOP 15 by TOTAL VAEP (final) ===
   display_name  total_minutes  actions  total_vaep  vaep_per_90
  Kylian Mbappé            686      658    6.041780     0.792653
   Lionel Messi            786      873    5.354761     0.613141
 Julián Álvarez            512      293    2.531266     0.444949
 Joško Gvardiol            749     1033    2.407241     0.289255
 Theo Hernández            571      656    2.258241     0.355940
    Bukayo Saka            311      265    2.077206     0.601121
 Enzo Fernandez            643      933    1.986203     0.278007
     Harry Kane            447      245    1.856127     0.373717
     Cody Gakpo            484      353    1.758527     0.326999
     Nathan Aké            532      681    1.751598     0.296323
 E

## Diagnosis

In [ ]:
# Diagnostic 1: What does Messi's action breakdown look like?
messi_id = 5503

messi_actions = all_actions[all_actions['player_id'] == messi_id].copy()
print(f"Messi total actions: {len(messi_actions)}")
print(f"\nMessi action types:")
print(messi_actions['type_name'].value_counts())

print(f"\nMessi shots and their outcomes:")
messi_shots = messi_actions[messi_actions['type_name'].str.contains('shot', case=False, na=False)]
print(messi_shots[['type_name', 'result_name', 'start_x', 'start_y', 'P_scores', 'P_concedes', 'vaep_value']].to_string(index=False))

Messi total actions: 873

Messi action types:
type_name
dribble             402
pass                329
take_on              36
shot                 24
bad_touch            15
corner_crossed       14
freekick_short       12
foul                 11
shot_penalty          7
freekick_crossed      5
corner_short          5
tackle                5
cross                 4
shot_freekick         3
clearance             1
Name: count, dtype: int64

Messi shots and their outcomes:
    type_name result_name  start_x  start_y  P_scores  P_concedes  vaep_value
         shot        fail  20.9125   42.330  0.013172    0.002604    0.002766
         shot        fail  14.4375   26.945  0.030966    0.002708    0.017047
shot_freekick        fail  21.4375   27.880  0.030784    0.002362    0.028422
 shot_penalty     success  94.0625   34.425  0.782935    0.005647   -0.015164
         shot        fail  16.9750   44.880  0.010472    0.002069    0.000995
         shot        fail  20.9125   34.935  0.005363    

In [ ]:
# Diagnostic 2: Are goals being attributed correctly in general?
# Find all goals across the tournament and see who gets the VAEP
goals = all_actions[
    (all_actions['type_name'].str.contains('shot', case=False, na=False)) &
    (all_actions['result_name'] == 'success')
].copy()

print(f"Total successful shots (goals): {len(goals)}")
print(f"\nVAEP distribution for goals:")
print(goals['vaep_value'].describe())

# How many goals have low VAEP?
print(f"\nGoals with VAEP < 0.1: {(goals['vaep_value'] < 0.1).sum()}")
print(f"Goals with VAEP > 0.5: {(goals['vaep_value'] > 0.5).sum()}")

# Top-VAEP goals (should be 0.8+, the actual goal moments)
print(f"\nTop 10 VAEP goals:")
top_goals = goals.nlargest(10, 'vaep_value')[['type_name', 'result_name', 'P_scores', 'P_concedes', 'vaep_value']]
print(top_goals.to_string(index=False))

Total successful shots (goals): 195

VAEP distribution for goals:
count    195.000000
mean       0.376542
std        0.401967
min       -0.693581
25%        0.354547
50%        0.540408
75%        0.640083
max        0.766689
Name: vaep_value, dtype: float64

Goals with VAEP < 0.1: 43
Goals with VAEP > 0.5: 119

Top 10 VAEP goals:
type_name result_name  P_scores  P_concedes  vaep_value
     shot     success  0.788997    0.011697    0.766689
     shot     success  0.786653    0.005346    0.762268
     shot     success  0.749300    0.003019    0.742456
     shot     success  0.752918    0.004178    0.741928
     shot     success  0.790559    0.005939    0.729514
     shot     success  0.781973    0.009974    0.722954
     shot     success  0.751101    0.010096    0.718772
     shot     success  0.767174    0.002773    0.717432
     shot     success  0.766503    0.008755    0.712378
     shot     success  0.736327    0.005365    0.708726


In [ ]:
# Diagnostic 3: Who scored the most VAEP-valued goals?
goals_with_player = goals.merge(players[['player_id', 'player_name', 'nickname']], on='player_id', how='left')
goals_with_player['display_name'] = goals_with_player.apply(
    lambda r: r['nickname'] if pd.notna(r['nickname']) and r['nickname'] else r['player_name'],
    axis=1
)

top_goal_scorers_by_vaep = (
    goals_with_player
    .groupby('display_name')
    .agg(goals=('vaep_value', 'count'), total_goal_vaep=('vaep_value', 'sum'))
    .sort_values('total_goal_vaep', ascending=False)
    .head(20)
)
print(top_goal_scorers_by_vaep)

                        goals  total_goal_vaep
display_name                                  
Kylian Mbappé               9         2.959951
Olivier Giroud              4         2.182751
Gonçalo Ramos               3         2.130220
Julián Álvarez              4         2.053846
Álvaro Morata               3         1.918366
Cody Gakpo                  3         1.832808
Bukayo Saka                 3         1.802193
Richarlison                 3         1.772679
Marcus Rashford             3         1.547393
Niclas Füllkrug             2         1.383976
Andrej Kramarić             2         1.349733
Youssef En-Nesyri           2         1.301943
Vincent Aboubakar           2         1.299095
Kai Havertz                 2         1.267607
Giorgian De Arrascaeta      2         1.244529
Mohammed Kudus              2         1.117886
Enner Valencia              3         1.102492
Rafael Leão                 2         1.097313
Aleksandar Mitrović         2         1.061165
Salem Al Daws

In [ ]:
all_penalties = all_actions[all_actions['type_name'] == 'shot_penalty'].copy()
print(f"Total penalties: {len(all_penalties)}")
print(f"\nPenalty start_x distribution:")
print(all_penalties['start_x'].describe())
print(f"\nPenalties at start_x < 50 (incorrectly placed): {(all_penalties['start_x'] < 50).sum()}")
print(f"Penalties at start_x >= 50 (correctly placed): {(all_penalties['start_x'] >= 50).sum()}")

print(f"\nVAEP by penalty position:")
print(all_penalties.groupby(all_penalties['start_x'] < 50)['vaep_value'].describe())

Total penalties: 64

Penalty start_x distribution:
count    64.000000
mean     94.108984
std       0.046689
min      93.975000
25%      94.062500
50%      94.150000
75%      94.150000
max      94.150000
Name: start_x, dtype: float64

Penalties at start_x < 50 (incorrectly placed): 0
Penalties at start_x >= 50 (correctly placed): 64

VAEP by penalty position:
         count      mean       std       min      25%       50%       75%  \
start_x                                                                     
False     64.0 -0.528527  0.380986 -1.173587 -0.76712 -0.536125 -0.068086   

              max  
start_x            
False   -0.004818  


In [ ]:
# Check 1: Messi's penalties should now have positive VAEP
messi_id = 5503
messi_pens = all_actions[
    (all_actions['player_id'] == messi_id) &
    (all_actions['type_name'] == 'shot_penalty')
]
print("Messi's penalties (post-fix):")
print(messi_pens[['type_name', 'result_name', 'start_x', 'P_scores', 'P_concedes', 'vaep_value']].to_string(index=False))

# Check 2: Where does Messi rank now?
total_rank = qualified.sort_values('total_vaep', ascending=False).reset_index(drop=True)
messi_row = total_rank[total_rank['display_name'].str.contains('Messi', na=False)]
print(f"\nMessi rank by total VAEP: {messi_row.index.values + 1}")
print(messi_row[['display_name', 'total_minutes', 'total_vaep', 'vaep_per_90']])

# Check 3: Top 15 after the fix
print("\n=== TOP 15 by TOTAL VAEP (after penalty fix) ===")
print(
    total_rank
    .head(15)[['display_name', 'total_minutes', 'actions', 'total_vaep', 'vaep_per_90']]
    .to_string(index=False)
)

Messi's penalties (post-fix):
   type_name result_name  start_x  P_scores  P_concedes  vaep_value
shot_penalty     success  94.0625  0.782935    0.005647   -0.015164
shot_penalty     success  94.1500  0.756559    0.657687   -0.693581
shot_penalty     success  94.0625  0.779079    0.010743   -0.024117
shot_penalty     success  94.1500  0.756559    0.643275   -0.679169
shot_penalty        fail  94.1500  0.029383    0.007027   -0.770097
shot_penalty     success  94.0625  0.797078    0.022912   -0.018287
shot_penalty     success  94.0625  0.737396    0.014309   -0.069367

Messi rank by total VAEP: [211]
     display_name  total_minutes  total_vaep  vaep_per_90
210  Lionel Messi            786    0.094979     0.010875

=== TOP 15 by TOTAL VAEP (after penalty fix) ===
   display_name  total_minutes  actions  total_vaep  vaep_per_90
  Kylian Mbappé            686      658    3.964861     0.520171
 Julián Álvarez            512      293    2.531266     0.444949
 Joško Gvardiol            749  